In [12]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical


In [13]:
path = r'/content/drive/MyDrive/Colab Notebooks/raw_pe_images.csv'
try:
    print("Processing.....\n")
    df = pd.read_csv(path)
    print("Dataset Loaded successfully......\n")
except FileExistsError:
    print("Please fix the path or Dataset does not exist")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


Processing.....

Dataset Loaded successfully......



In [14]:
print(f"The size of the dataset :{df.shape}\n")
print(f"The features column:{df.columns.tolist()}\n")
print(f"Dataset Information :{df.info()}\n")
print(f"Sample Data :{df.sample(5)}\n")
print(f"Summary of the Dataset :{df.describe}\n")


The size of the dataset :(51959, 1026)

The features column:['hash', 'pix_0', 'pix_1', 'pix_2', 'pix_3', 'pix_4', 'pix_5', 'pix_6', 'pix_7', 'pix_8', 'pix_9', 'pix_10', 'pix_11', 'pix_12', 'pix_13', 'pix_14', 'pix_15', 'pix_16', 'pix_17', 'pix_18', 'pix_19', 'pix_20', 'pix_21', 'pix_22', 'pix_23', 'pix_24', 'pix_25', 'pix_26', 'pix_27', 'pix_28', 'pix_29', 'pix_30', 'pix_31', 'pix_32', 'pix_33', 'pix_34', 'pix_35', 'pix_36', 'pix_37', 'pix_38', 'pix_39', 'pix_40', 'pix_41', 'pix_42', 'pix_43', 'pix_44', 'pix_45', 'pix_46', 'pix_47', 'pix_48', 'pix_49', 'pix_50', 'pix_51', 'pix_52', 'pix_53', 'pix_54', 'pix_55', 'pix_56', 'pix_57', 'pix_58', 'pix_59', 'pix_60', 'pix_61', 'pix_62', 'pix_63', 'pix_64', 'pix_65', 'pix_66', 'pix_67', 'pix_68', 'pix_69', 'pix_70', 'pix_71', 'pix_72', 'pix_73', 'pix_74', 'pix_75', 'pix_76', 'pix_77', 'pix_78', 'pix_79', 'pix_80', 'pix_81', 'pix_82', 'pix_83', 'pix_84', 'pix_85', 'pix_86', 'pix_87', 'pix_88', 'pix_89', 'pix_90', 'pix_91', 'pix_92', 'pix_93', '

In [15]:
X = df.drop(columns=['hash','malware'])
Y = df['malware']
print(f"Input shape of X:{X.shape}\n")
print(f"Output shape of Y: {Y.shape}\n")

Input shape of X:(51959, 1024)

Output shape of Y: (51959,)



In [16]:
X_scaled = X.values / 255
X_scaled = X_scaled.reshape(-1, 32, 32, 1)

In [17]:
X_train,X_test,Y_train,Y_test = train_test_split(X_scaled,Y,test_size=0.2,random_state=42)

In [18]:
gpus = len(tf.config.list_physical_devices('GPU'))

if(gpus > 0):
    print(f"GPU Detected :{gpus}")
    print("Training via GPU")

    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("GPU not found !!!")
    print("Training via CPU")

GPU not found !!!
Training via CPU


In [19]:
model = Sequential([
    Input(shape = (32,32,1)),

    Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same'),
    MaxPooling2D(pool_size=(2,2)),

    Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'),
    MaxPooling2D(pool_size=(2,2)),

    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │       262,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 281,089 (1.07 MB)

 Trainable params: 281,089 (1.07 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
history = model.fit(X_train,Y_train,epochs=100, batch_size=32, validation_data=(X_test, Y_test))

Epoch 1/100
1299/1299 ━━━━━━━━━━━━━━━━━━━━ 65s 49ms/step - accuracy: 0.9498 - loss: 0.1836 - val_accuracy: 0.9506 - val_loss: 0.1556
Epoch 2/100
1299/1299 ━━━━━━━━━━━━━━━━━━━━ 65s 50ms/step - accuracy: 0.9502 - loss: 0.1567 - val_accuracy: 0.9506 - val_loss: 0.1525
Epoch 3/100
1299/1299 ━━━━━━━━━━━━━━━━━━━━ 80s 49ms/step - accuracy: 0.9502 - loss: 0.1448 - val_accuracy: 0.9506 - val_loss: 0.1493
Epoch 4/100
1299/1299 ━━━━━━━━━━━━━━━━━━━━ 64s 49ms/step - accuracy: 0.9502 - loss: 0.1362 - val_accuracy: 0.9506 - val_loss: 0.1400
Epoch 5/100
1299/1299 ━━━━━━━━━━━━━━━━━━━━ 63s 48ms/step - accuracy: 0.9502 - loss: 0.1285 - val_accuracy: 0.9506 - val_loss: 0.1371
Epoch 6/100
1299/1299 ━━━━━━━━━━━━━━━━━━━━ 62s 48ms/step - accuracy: 0.9502 - loss: 0.1206 - val_accuracy: 0.9506 - val_loss: 0.1411
Epoch 7/100
1299/1299 ━━━━━━━━━━━━━━━━━━━━ 65s 50ms/step - accuracy: 0.9502 - loss: 0.1118 - val_accuracy: 0.9506 - val_loss: 0.1429
Epoch 8/100
1299/1299 ━━━━━━━━━━━━━━━━━━━━ 80s 49ms/step - accuracy: 

# Evaluate the Model

In [21]:
val_loss, val_accuracy = model.evaluate(X_test, Y_test)
print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")

325/325 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9523 - loss: 1.2466
Validation Accuracy: 95.23%


# Apply threshold (0.5) for binary classification (0 = Benign, 1 = Malware)

In [22]:
sample = X_test[0].reshape(1, 32, 32, 1)

prediction_prob = model.predict(sample)[0][0]
predicted_label = 1 if prediction_prob >= 0.5 else 0
print(f"Prediction Probability: {prediction_prob:.4f}")
print(f"Predicted Label: {predicted_label} ('1' = Malware, '0' = Benign)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step
Prediction Probability: 1.0000
Predicted Label: 1 ('1' = Malware, '0' = Benign)


In [25]:
print("Saving the model")
# Use the recommended native .keras extension
model.save("malware_cnn_model.keras")

Saving the model
